# Lab 10: Backtesting and Validation

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab10_backtesting.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Objective

This lab makes selection bias visible. You will create 200 strategies
that have no population edge, select the best historical Sharpe, and
test whether that champion travels across other parts of the same
history. You will then add a known weak edge and ask whether the
diagnostics move in the direction they should.

The simulation gives us a rare advantage: we know the data-generating
process. Every impressive result in the first garden is sampling luck.
That makes it possible to distinguish what a statistic measures from the
story we might otherwise tell about it.

**Time estimate:** 90–120 minutes

After the lab, you should be able to:

-   Explain why a selected Sharpe is the maximum of a search
-   Separate temporal walk-forward validation from CSCV
-   Interpret PBO as rank instability within a declared candidate garden
-   Compare PSR against zero with DSR against a search-adjusted
    benchmark
-   State the trial-count and dependence assumptions behind DSR
-   Keep statistical search, implementation costs, and investment
    decisions separate

> **Assessment boundary**
>
> Walk-forward evaluation remains central when the coursework scaffold
> makes sequential predictions. HAC inference remains central when you
> estimate a pre-specified factor regression. PBO and DSR are
> appropriate when you have searched a garden of alternative strategies.
> This lab teaches those extensions; it does not add an undocumented
> requirement to the assessment brief.

## Setup

The helper functions live in `src/fdslab/validation.py`;
`scripts/utilities/overfit_metrics.py` is the compatibility import used
by the chapter and Colab.

In [ ]:
import sys
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for candidate in (
    Path("scripts/utilities").resolve(),
    (Path.cwd().parent / "scripts" / "utilities").resolve(),
):
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break

try:
    from overfit_metrics import (
        cscv_pbo,
        deflated_sharpe_ratio,
        expected_max_sharpe,
        generate_noise_strategies,
        probabilistic_sharpe_ratio,
        sharpe_ratio,
    )
except ModuleNotFoundError:
    helper_url = (
        "https://raw.githubusercontent.com/quinfer/"
        "financial-data-science/main/src/fdslab/validation.py"
    )
    urllib.request.urlretrieve(helper_url, "overfit_metrics.py")
    from overfit_metrics import (
        cscv_pbo,
        deflated_sharpe_ratio,
        expected_max_sharpe,
        generate_noise_strategies,
        probabilistic_sharpe_ratio,
        sharpe_ratio,
    )

ANNUALISE = np.sqrt(12)

## Part 1: Name the validation question

Walk-forward and combinatorially symmetric cross-validation (CSCV) use
time blocks for different purposes.

**Walk-forward** asks: if the model had been refitted using information
available at each date, how would the sequence of next-period forecasts
have performed? Its test observations follow a deployment chronology.
Each preprocessing and tuning step belongs inside the training window.

**CSCV** asks: when the best candidate is chosen in one balanced half of
the history, where does that same candidate rank in the complementary
half? It recombines contiguous slices symmetrically. It estimates
selection instability, not a live trading path (Bailey et al. 2015).

Classify each design before writing code:

1.  Fit through December 2018, predict January 2019, expand by one
    month, and repeat.
2.  Divide twenty years into ten slices, choose five as in sample, and
    use the other five out of sample for each symmetric combination.
3.  Shuffle months into five random folds before estimating a next-month
    return model.
4.  Fit on 2000–2018, inspect 2019–2023, revise the lookback, and score
    2019–2023 again.

## Part 2: Build a noise garden

We generate 240 monthly observations for 200 strategies. Every strategy
has population mean zero. Their pairwise correlation is 0.2, so the
garden resembles related parameter variants rather than 200 unrelated
bets.

In [ ]:
T = 240
M = 200
RHO = 0.2
SEED = 42

noise = generate_noise_strategies(T=T, N=M, rho=RHO, seed=SEED)

mean_pairwise_corr = (
    pd.DataFrame(noise).corr().to_numpy()[np.triu_indices(M, k=1)].mean()
)
trial_sharpes = np.array([sharpe_ratio(noise[:, j]) for j in range(M)])
winner = int(np.argmax(trial_sharpes))

garden_audit = pd.Series({
    "Months": T,
    "Strategies": M,
    "Population mean of every strategy": 0.0,
    "Target pairwise correlation": RHO,
    "Realised mean pairwise correlation": mean_pairwise_corr,
    "Winning monthly Sharpe": trial_sharpes[winner],
    "Winning annualised Sharpe": trial_sharpes[winner] * ANNUALISE,
})

print("=== Noise garden audit ===")
print(garden_audit.round(4))

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.hist(trial_sharpes * ANNUALISE, bins=28, color="steelblue", alpha=0.85)
ax.axvline(
    trial_sharpes[winner] * ANNUALISE,
    color="coral",
    linestyle="--",
    linewidth=2,
    label=f"Selected winner: {trial_sharpes[winner] * ANNUALISE:.3f}",
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Annualised sample Sharpe")
ax.set_ylabel("Strategies")
ax.set_title("The winner of 200 zero-edge strategies")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

Answer from the output:

1.  Why is the realised mean pairwise correlation close to, but not
    exactly, 0.2?
2.  How can the selected strategy have a positive Sharpe when every
    population mean is zero?
3.  What information would be lost if the researcher showed only the
    coral line?

## Part 3: Price the maximum

The false-strategy benchmark $SR^*$ is the expected maximum Sharpe under
a zero-edge null. It rises with the number of independent trials and
with the dispersion of Sharpe estimates (Bailey and Prado 2014).

First treat the 200 sample Sharpes as independent to isolate the
trial-count effect. We will incorporate dependence in Part 5.

In [ ]:
sigma_sr = trial_sharpes.std(ddof=1)
trial_grid = np.array([2, 5, 10, 20, 50, 100, 200, 500, 1000])
sr_star_grid = expected_max_sharpe(trial_grid, sigma_sr)

benchmark_table = pd.DataFrame({
    "Independent trials": trial_grid,
    "Expected maximum monthly Sharpe": sr_star_grid,
    "Expected maximum annualised Sharpe": sr_star_grid * ANNUALISE,
})

print("=== Expected zero-edge winner ===")
print(benchmark_table.round(3).to_string(index=False))

1.  Does doubling the number of trials double $SR^*$?
2.  Why does a short or noisy backtest have a larger $SR^*$ than a long,
    stable one with the same number of trials?
3.  If several strategy variants are tried informally and discarded, do
    they belong in the trial count?

## Part 4: Test rank stability

CSCV divides the 240 months into ten contiguous slices. For each
selected combination of five slices, it chooses the highest-Sharpe
strategy in sample and records the same strategy’s Sharpe and rank in
the complementary five slices.

There are $\binom{10}{5}=252$ symmetric splits. We use a fixed subsample
of 150 to keep the lab quick. The helper’s split sampler uses seed 42,
so the result is reproducible.

In [ ]:
noise_cscv = cscv_pbo(noise, n_folds=10, max_splits=150)

annual_is = noise_cscv.sr_is * ANNUALISE
annual_oos = noise_cscv.sr_oos * ANNUALISE
slope, intercept = np.polyfit(annual_is, annual_oos, 1)

print("=== CSCV on the noise garden ===")
print(f"Splits used: {noise_cscv.splits_used}")
print(f"PBO: {noise_cscv.pbo:.3f}")
print(f"Share of champion OOS Sharpes below zero: {(annual_oos < 0).mean():.3f}")
print(f"Slope of champion OOS Sharpe on IS Sharpe: {slope:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(annual_is, annual_oos, s=22, alpha=0.7, color="steelblue")
x_line = np.linspace(annual_is.min(), annual_is.max(), 80)
axes[0].plot(x_line, slope * x_line + intercept, color="coral", linewidth=2)
axes[0].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Champion in-sample Sharpe")
axes[0].set_ylabel("Same strategy out-of-sample Sharpe")
axes[0].set_title("Performance degradation")

axes[1].hist(noise_cscv.taus, bins=20, color="steelblue", alpha=0.85)
axes[1].axvline(0, color="coral", linestyle="--", linewidth=2)
axes[1].set_xlabel("Logit of out-of-sample rank")
axes[1].set_ylabel("CSCV splits")
axes[1].set_title("Rank stability")

plt.tight_layout()
plt.show()

PBO is the fraction of splits in which the in-sample champion finishes
below the out-of-sample median. In the logit panel, those are the
observations to the left of zero.

1.  Why should exchangeable noise produce PBO near 0.5?
2.  What does the fitted slope say about the most impressive in-sample
    champions in this draw?
3.  Why is this figure not a walk-forward equity curve?
4.  Can PBO tell whether the out-of-sample return is positive after
    costs?

## Part 5: Raise the Sharpe benchmark

The probabilistic Sharpe ratio (PSR) compares the selected Sharpe with a
chosen benchmark and adjusts its uncertainty for sample length,
skewness, and kurtosis. Testing the winner against zero is weak because
the maximum of 200 zero-edge estimates is usually positive.

The deflated Sharpe ratio (DSR) uses the expected maximum under the
search as the benchmark. For $M$ correlated trials, Appendix A.3 of
Bailey and Prado (2014) approximates the effective number of independent
trials as:

$$
N_{\mathrm{eff}}=\rho+(1-\rho)M.
$$

The course helper uses this approximation and the observed cross-trial
Sharpe dispersion.

In [ ]:
winner_returns = noise[:, winner]
winner_skew = float(pd.Series(winner_returns).skew())
winner_kurtosis = float(pd.Series(winner_returns).kurtosis() + 3.0)
winner_sr = trial_sharpes[winner]

psr_zero = probabilistic_sharpe_ratio(
    sr_hat=winner_sr,
    sr_benchmark=0.0,
    n_obs=T,
    skew=winner_skew,
    kurtosis=winner_kurtosis,
)

dsr_noise, sr_star_noise = deflated_sharpe_ratio(
    sr_hat=winner_sr,
    sr_trials=trial_sharpes,
    n_obs=T,
    skew=winner_skew,
    kurtosis=winner_kurtosis,
    rho=RHO,
)

n_effective = RHO + (1.0 - RHO) * M

selection_table = pd.Series({
    "Selected annualised Sharpe": winner_sr * ANNUALISE,
    "Effective trial count": n_effective,
    "Annualised selection benchmark SR*": sr_star_noise * ANNUALISE,
    "PSR against zero": psr_zero,
    "DSR against SR*": dsr_noise,
    "PSR minus DSR": psr_zero - dsr_noise,
})

print("=== Selection-adjusted Sharpe evidence ===")
print(selection_table.round(3))

1.  Why can PSR against zero be near one in a garden known to contain no
    edge?
2.  What does the PSR minus DSR gap measure in this design?
3.  Why must the report state $M$, $\rho$, and the Sharpe dispersion
    rather than giving DSR alone?
4.  What limitation remains in the average-correlation approximation?

## Part 6: Add a known edge

We now add a population annualised Sharpe of about 0.5 to the first ten
strategies. The other 190 retain zero edge. This is a controlled
contrast, not a claim that a live factor has exactly this distribution.

In [ ]:
edge = generate_noise_strategies(T=T, N=M, rho=RHO, seed=SEED)
edge[:, :10] += 0.50 / ANNUALISE

edge_sharpes = np.array([sharpe_ratio(edge[:, j]) for j in range(M)])
edge_winner = int(np.argmax(edge_sharpes))
edge_cscv = cscv_pbo(edge, n_folds=10, max_splits=150)

edge_returns = edge[:, edge_winner]
edge_skew = float(pd.Series(edge_returns).skew())
edge_kurtosis = float(pd.Series(edge_returns).kurtosis() + 3.0)
edge_psr = probabilistic_sharpe_ratio(
    edge_sharpes[edge_winner], 0.0, T, edge_skew, edge_kurtosis
)
edge_dsr, edge_sr_star = deflated_sharpe_ratio(
    sr_hat=edge_sharpes[edge_winner],
    sr_trials=edge_sharpes,
    n_obs=T,
    skew=edge_skew,
    kurtosis=edge_kurtosis,
    rho=RHO,
)

comparison = pd.DataFrame({
    "Noise garden": [
        winner_sr * ANNUALISE,
        noise_cscv.pbo,
        psr_zero,
        sr_star_noise * ANNUALISE,
        dsr_noise,
    ],
    "Weak-edge garden": [
        edge_sharpes[edge_winner] * ANNUALISE,
        edge_cscv.pbo,
        edge_psr,
        edge_sr_star * ANNUALISE,
        edge_dsr,
    ],
}, index=[
    "Selected annualised Sharpe",
    "PBO",
    "PSR against zero",
    "Annualised SR*",
    "DSR",
])

print("=== Known-truth comparison ===")
print(comparison.round(3))
print(f"Weak-edge winner is one of the ten edged strategies: {edge_winner < 10}")

1.  Did PBO and DSR move in the expected direction?
2.  Why did the metrics improve without becoming one?
3.  What would happen if the same mean shift were spread across all 200
    strategies?
4.  Why does detecting an edge still not establish an investable
    strategy?

## Part 7: Write the research decision

Write a 150-word memo about the **noise garden**. Use the actual outputs
from your run and include:

-   The candidate count, history length, and correlation assumption
-   The selected annualised Sharpe
-   PBO and what its denominator is
-   PSR against zero, DSR, and the selection-adjusted $SR^*$
-   A decision to reject, continue researching, or paper trade
-   One limitation of the simulation

Then add one sentence explaining what changed in the weak-edge garden
without claiming that the exercise proves those ten strategies would
work live.

## Reporting template

For a real strategy garden, report:

-   **Data:** universe, sample, point-in-time treatment, label horizon,
    and costs
-   **Search:** every trialled configuration, selection metric, and
    failed variants
-   **Chronology:** walk-forward design, window length, purge or embargo
    where labels overlap
-   **CSCV:** number of slices and splits, PBO, logit distribution, and
    out-of-sample loss rate
-   **Sharpe evidence:** PSR benchmark, $M$, estimated $\rho$, effective
    $N$, $SR^*$, and DSR
-   **Decision:** thresholds fixed before seeing the outputs and the
    action they imply

A single pre-specified factor regression does not need a manufactured
garden of variants. Report HAC inference and appropriate stability
checks for that estimand. A strategy chosen from many alternatives does
need the search recorded next to its winner.

## References

Bailey, David H., Jonathan M. Borwein, Marcos López de Prado, and Qiji
Jim Zhu. 2015. “The Probability of Backtest Overfitting.” *Journal of
Computational Finance*. <https://doi.org/10.2139/ssrn.2326253>.

Bailey, David H., and Marcos López de Prado. 2014. “The Deflated Sharpe
Ratio: Correcting for Selection Bias, Backtest Overfitting and
Non-Normality.” *Journal of Portfolio Management* 40 (5): 94–107.
<https://doi.org/10.2139/ssrn.2460551>.